# Baseline Benchmark: Single-Shot Anthropic Summaries

Generates one-shot clinical note summaries using the Anthropic API and scores
them with the same metrics as the EHR pipeline benchmark, enabling a
head-to-head comparison.


In [41]:
import asyncio
import importlib
import json
import logging
import os
import random
import sys
import time
import warnings
from pathlib import Path
import pandas as pd
%pip install anthropic
import anthropic

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
warnings.filterwarnings("ignore", category=FutureWarning)

import benchmarks.mimic
import benchmarks.metrics
importlib.reload(benchmarks.mimic)
importlib.reload(benchmarks.metrics)
from benchmarks import mimic, metrics

print("project root:", PROJECT_ROOT)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
project root: /Users/natejly/Documents/GitHub/LLMS


## Configuration

In [42]:
# load .env file
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
assert ANTHROPIC_API_KEY, "Set ANTHROPIC_API_KEY as an environment variable before running."

MODEL = "claude-opus-4-7"
MAX_CONCURRENT = 10

EHR_JSON_PATH = PROJECT_ROOT / "datasets" / "bquxjob_5dbe3bd2_19dae30f3cf.json"
NOTES_CSV_PATH = PROJECT_ROOT / "datasets" / "mimic-iii-ext-notes-1.0.0" / "notes.csv"
LABELS_CSV_PATH = PROJECT_ROOT / "datasets" / "mimic-iii-ext-notes-1.0.0" / "labels.csv"

BASELINE_DIR = PROJECT_ROOT / "outputs" / "_baseline"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

NUM_CASES = 100
RANDOM_SEED = 7
SUMMARY_MIN_RATIO = 0.20
SUMMARY_MAX_RATIO = 0.30
BERTSCORE_MODEL = "roberta-large"
RESUME = True

client = anthropic.AsyncAnthropic(api_key=ANTHROPIC_API_KEY)

print(f"model: {MODEL}")
print(f"max concurrent requests: {MAX_CONCURRENT}")
print(f"work dir: {BASELINE_DIR}")
print(f"compression target: {SUMMARY_MIN_RATIO:.0%}-{SUMMARY_MAX_RATIO:.0%}")
print(f"resume from cache: {RESUME}")

model: claude-opus-4-7
max concurrent requests: 10
work dir: /Users/natejly/Documents/GitHub/LLMS/outputs/_baseline
compression target: 20%-30%
resume from cache: True


## Load and sample cases (same seed / sample as pipeline benchmark)

In [43]:
all_cases = mimic.load_real_cases(
    ehr_json_path=EHR_JSON_PATH,
    notes_csv_path=NOTES_CSV_PATH,
    labels_csv_path=LABELS_CSV_PATH,
)
print(f"loaded {len(all_cases)} cases")

rng = random.Random(RANDOM_SEED)
sampled = rng.sample(all_cases, k=min(NUM_CASES, len(all_cases)))
print(f"sampled {len(sampled)} cases (seed={RANDOM_SEED})")
for c in sampled[:5]:
    print(f"  {c.case_id:36s}  note_chars={len(c.note_text):5d}  gold_concepts={len(c.gold_concepts):3d}")
if len(sampled) > 5:
    print(f"  ... and {len(sampled) - 5} more")

loaded 150 cases
sampled 100 cases (seed=7)
  mimic-72907-165405-520384             note_chars= 3404  gold_concepts=  9
  mimic-29307-175627-318233             note_chars=  784  gold_concepts=  7
  mimic-47927-161682-717576             note_chars= 2592  gold_concepts=  7
  mimic-77013-141363-444743             note_chars= 2827  gold_concepts=  7
  mimic-54229-165594-598031             note_chars= 2337  gold_concepts=  9
  ... and 95 more


## Generate single-shot summaries via Anthropic

In [44]:
SYSTEM_PROMPT = """You are a senior clinician's documentation assistant.

Your task: given a raw clinical note, produce a concise, accurate,
clinician-facing EHR summary in GitHub-flavored Markdown.

Hard constraints (violations are patient-safety issues):
- Include ONLY facts explicitly stated in the note. Never infer, assume,
  or add clinical information that is not directly present.
- Do not fabricate medications, doses, lab values, vital signs, or dates.
- If a finding is hedged in the note ("possible", "rule out",
  "consider"), preserve that hedge verbatim in the summary.
- Do not editorialize, add differential diagnoses, or offer management
  recommendations beyond what the note states.

Required structure (use these exact H2 headings in this order):

## HPI
One or two sentences: age, sex, chief complaint, relevant PMH, and
presenting course. If the note does not contain this information, write
"_Not documented._"

## Active Problems
Dense prose (not bullet points). Group related diagnoses into sentences.
Name every diagnosis, condition, and clinical finding mentioned in the
note. Include severity, staging, or qualifiers when present. Separate
major organ-system groups with new sentences (e.g., cardiac, renal,
infectious, respiratory, chronic conditions, surgical history).

## Medications
Prose or short comma-separated list. Include drug names, doses, routes,
and frequencies when available in the note.

## Labs
Prose or short comma-separated list. Include lab names with numeric
values and units. Include imaging and diagnostic test results (e.g.,
CT findings, culture results, EKG).

## Plan
Prose or short comma-separated list. Include planned or ongoing
interventions, pending workup, and disposition.

If a section has no relevant information in the note, write
"_Not documented._" for that section.

Style: short sentences, telegraphic clinical tone, standard medical
abbreviations are acceptable (e.g. HTN, DM2, CHF, AFib). No greetings,
disclaimers, or patient identifiers beyond what is in the note.
Do NOT include inline citations or reference tags of any kind.

Stay within the character budget specified by the user."""


def _user_prompt(note_text: str, min_chars: int, max_chars: int) -> str:
    return (
        f"Summarize the following clinical note. "
        f"Target length: {min_chars}\u2013{max_chars} characters. "
        f"Use the exact section structure from your instructions.\n\n"
        f"--- BEGIN CLINICAL NOTE ---\n{note_text}\n--- END CLINICAL NOTE ---"
    )


async def summarize_one(
    case: mimic.RealCase,
    semaphore: asyncio.Semaphore,
) -> dict:
    case_dir = BASELINE_DIR / case.case_id
    cache_path = case_dir / "baseline_summary.md"
    meta_path = case_dir / "baseline_meta.json"

    if RESUME and cache_path.exists() and meta_path.exists():
        summary = cache_path.read_text(encoding="utf-8")
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        return {"case_id": case.case_id, "summary": summary, **meta}

    note_chars = max(len(case.note_text), 1)
    min_chars = max(int(note_chars * SUMMARY_MIN_RATIO), 200)
    max_chars = max(int(note_chars * SUMMARY_MAX_RATIO), min_chars + 100)

    async with semaphore:
        t0 = time.monotonic()
        resp = await client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": _user_prompt(case.note_text, min_chars, max_chars)},
            ],
        )
        elapsed = time.monotonic() - t0

    summary = resp.content[0].text.strip() if resp.content else ""
    usage = resp.usage

    case_dir.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(summary + "\n", encoding="utf-8")
    meta = {
        "model": MODEL,
        "elapsed_s": round(elapsed, 2),
        "input_tokens": usage.input_tokens if usage else 0,
        "output_tokens": usage.output_tokens if usage else 0,
        "note_chars": note_chars,
        "summary_chars": len(summary),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    return {"case_id": case.case_id, "summary": summary, **meta}


def _normalize(r):
    r["prompt_tokens"] = r.get("input_tokens", 0)
    r["completion_tokens"] = r.get("output_tokens", 0)
    return r


_progress_lock = asyncio.Lock()
_progress_done = 0


async def _summarize_tracked(case, sem, total):
    global _progress_done
    result = await summarize_one(case, sem)
    async with _progress_lock:
        _progress_done += 1
        if _progress_done % 10 == 0 or _progress_done == total:
            print(f"  [{_progress_done}/{total}] completed")
    return result


async def run_all(cases):
    global _progress_done
    _progress_done = 0
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    total = len(cases)
    print(f"launching {total} requests (max {MAX_CONCURRENT} concurrent)...")
    tasks = [_summarize_tracked(c, sem, total) for c in cases]
    return await asyncio.gather(*tasks, return_exceptions=True)


raw_results = await run_all(sampled)

errors = [r for r in raw_results if isinstance(r, Exception)]
successes = [_normalize(r) for r in raw_results if not isinstance(r, Exception)]
print(f"\ncompleted: {len(successes)}  errors: {len(errors)}")
for e in errors[:5]:
    print(f"  ERROR: {e}")

launching 100 requests (max 10 concurrent)...


2026-05-03 15:07:38,418 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:41,092 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:41,607 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:41,668 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:42,292 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:42,621 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:42,755 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:43,112 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:44,445 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2

  [10/100] completed


2026-05-03 15:07:48,088 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:48,970 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:49,153 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:49,769 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:50,621 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:51,855 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:52,792 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:53,379 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:07:54,004 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2

  [20/100] completed


...

  [80/100] completed


2026-05-03 15:08:53,606 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:53,722 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:53,840 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:53,886 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:54,490 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:56,703 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:57,660 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:08:59,933 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:00,120 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2

  [90/100] completed


2026-05-03 15:09:01,213 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:01,570 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:01,896 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:02,340 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:02,820 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:03,957 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:05,237 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:08,322 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-05-03 15:09:13,346 INFO httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2

  [100/100] completed

completed: 100  errors: 0


## Build references and score

In [45]:
case_lookup = {c.case_id: c for c in sampled}

results = []
for r in successes:
    case = case_lookup[r["case_id"]]
    reference = mimic.real_case_reference_summary(case)
    gold_ents = mimic.real_case_gold_entities(case)
    prediction = r["summary"]

    note_chars = r.get("note_chars", len(case.note_text))
    summary_chars = len(prediction)
    ratio = summary_chars / max(note_chars, 1)

    results.append({
        "case_id": r["case_id"],
        "prediction": prediction,
        "reference": reference,
        "gold_entities": gold_ents,
        "note_chars": note_chars,
        "summary_chars": summary_chars,
        "compression_ratio": ratio,
        "compression_in_band": SUMMARY_MIN_RATIO <= ratio <= SUMMARY_MAX_RATIO,
        "elapsed_s": r.get("elapsed_s", 0),
        "prompt_tokens": r.get("prompt_tokens", 0),
        "completion_tokens": r.get("completion_tokens", 0),
    })

print(f"{len(results)} cases ready for scoring")

100 cases ready for scoring


In [46]:
if results:
    r0 = results[0]
    print(f"--- {r0['case_id']} ---\n")
    print(f"compression: {r0['summary_chars']}/{r0['note_chars']} = {r0['compression_ratio']:.2f}\n")
    print("=== REFERENCE (first 1500 chars) ===\n")
    print(r0["reference"][:1500])
    print("\n=== BASELINE PREDICTION (first 1500 chars) ===\n")
    print(r0["prediction"][:1500])

--- mimic-72907-165405-520384 ---

compression: 1506/3404 = 0.44

=== REFERENCE (first 1500 chars) ===

## HPI
77.0-year-old female.
## Active Problems
- Abdominal Pain.
- Sepsis.
- Septicemia.
- Pain.
- Grimaces.
- Respiratory Failure.
- Kidney Failure.
- Atrial Fibrillation.
## EHR Diagnoses
- Other postoperative infection.
- Disseminated candidiasis.
- Severe sepsis.
- Septic shock.
- Other suppurative peritonitis.
- Acute and subacute necrosis of liver.
- Acute kidney failure with lesion of tubular necrosis.
- Necrotizing fasciitis.
- Acute vascular insufficiency of intestine.
- None.
- Acidosis.
- Acute salpingitis and oophoritis.
- Candidiasis of other urogenital sites.
- Acute inflammatory diseases of uterus, except cervix.
- Unspecified essential hypertension.
- Depressive disorder, not elsewhere classified.
- Unspecified acquired hypothyroidism.
- Other specified surgical operations and procedures causing abnormal patient reaction, or later complication, without mention of mis

## Compute ROUGE and entity metrics

In [47]:
rows = []
for r in results:
    if not r["prediction"].strip():
        continue
    row = {
        "case_id": r["case_id"],
        "compression_ratio": r["compression_ratio"],
        "compression_in_band": r["compression_in_band"],
        "elapsed_s": r["elapsed_s"],
        "prompt_tokens": r["prompt_tokens"],
        "completion_tokens": r["completion_tokens"],
    }
    row.update(metrics.rouge_scores(r["prediction"], r["reference"]))
    row.update(metrics.entity_recall_precision(
        prediction=r["prediction"],
        reference=r["reference"],
        gold_entities=r["gold_entities"],
    ))
    fk = metrics.flesch_kincaid_grade(r["prediction"])
    row["fk_grade"] = fk["fk_grade"]
    row["fk_in_target_band"] = fk["fk_in_target_band"]
    rows.append(row)

rouge_df = pd.DataFrame(rows)
rouge_df

2026-05-03 15:09:14,785 INFO absl: Using default tokenizer.
2026-05-03 15:09:14,816 INFO absl: Using default tokenizer.
2026-05-03 15:09:14,830 INFO absl: Using default tokenizer.
2026-05-03 15:09:14,864 INFO absl: Using default tokenizer.
2026-05-03 15:09:14,942 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,039 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,097 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,154 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,261 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,419 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,552 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,655 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,725 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,784 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,837 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,904 INFO absl: Using default tokenizer.
2026-05-03 15:09:15,986 INFO absl: Using

,case_id,compression_ratio,compression_in_band,elapsed_s,prompt_tokens,completion_tokens,rouge1_f,rouge2_f,rougeL_f,entity_precision,entity_recall,entity_f1,entity_tp,entity_fp,entity_fn,entity_gold,fk_grade,fk_in_target_band
0,mimic-72907-165405-520384,0.442421,False,8.73,2227,684,0.173494,0.029056,0.091566,1.0,0.195122,0.326531,8,0,33,41,14.20,False
1,mimic-29307-175627-318233,0.613520,False,4.27,1191,260,0.142077,0.044199,0.098361,1.0,0.130435,0.230769,3,0,20,23,9.26,False
2,mimic-47927-161682-717576,0.454090,False,7.47,1840,556,0.107023,0.006734,0.060201,1.0,0.137931,0.242424,4,0,25,29,9.64,False
3,mimic-77013-141363-444743,0.442519,False,8.97,1982,606,0.103020,0.014260,0.049734,1.0,0.128205,0.227273,5,0,34,39,8.89,True
4,mimic-54229-165594-598031,0.449722,False,8.47,1773,478,0.131673,0.032143,0.060498,1.0,0.250000,0.400000,9,0,27,36,9.38,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,mimic-83607-156758-631634,0.422043,False,8.52,1966,566,0.143791,0.013129,0.056645,1.0,0.093023,0.170213,4,0,39,43,10.56,False
96,mimic-25326-191645-386337,0.741040,False,5.16,1220,346,0.074074,0.011152,0.040741,1.0,0.066667,0.125000,3,0,42,45,11.26,False
97,mimic-56635-192782-481041,0.463421,False,13.42,2372,822,0.127737,0.018315,0.062044,1.0,0.122449,0.218182,6,0,43,49,10.89,False
98,mimic-68135-163192-486433,0.363146,False,3.86,1397,261,0.082759,0.013889,0.055172,1.0,0.060606,0.114286,2,0,31,33,8.98,True


## BERTScore (batched)

In [48]:
non_empty = [r for r in results if r["prediction"].strip()]
preds = [r["prediction"] for r in non_empty]
refs = [r["reference"] for r in non_empty]
case_ids = [r["case_id"] for r in non_empty]

if preds:
    bert_rows = metrics.bertscore_batch(preds, refs, model_type=BERTSCORE_MODEL)
    bert_df = pd.DataFrame(bert_rows, index=case_ids)
    bert_df.index.name = "case_id"
    bert_df = bert_df.reset_index()
else:
    bert_df = pd.DataFrame(columns=["case_id", "bertscore_p", "bertscore_r", "bertscore_f1"])

bert_df

2026-05-03 15:09:23,536 INFO httpx: HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-05-03 15:09:23,689 INFO httpx: HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-03 15:09:23,723 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-03 15:09:23,779 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-03 15:09:23,810 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-03 15:09:23,847 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main?recursive=true&expa

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,case_id,bertscore_p,bertscore_r,bertscore_f1
0,mimic-72907-165405-520384,0.779102,0.810328,0.794408
1,mimic-29307-175627-318233,0.809380,0.818459,0.813894
2,mimic-47927-161682-717576,0.784871,0.810171,0.797320
3,mimic-77013-141363-444743,0.772228,0.781443,0.776808
4,mimic-54229-165594-598031,0.770979,0.778451,0.774697
...,...,...,...,...
95,mimic-83607-156758-631634,0.787343,0.792782,0.790053
96,mimic-25326-191645-386337,0.784739,0.782490,0.783612
97,mimic-56635-192782-481041,0.776260,0.798569,0.787256
98,mimic-68135-163192-486433,0.797296,0.783993,0.790589


## Combined per-case metrics

In [49]:
if not rouge_df.empty and not bert_df.empty:
    full = rouge_df.merge(bert_df, on="case_id", how="left")
else:
    full = rouge_df.copy()

metric_cols = [c for c in (
    "rouge1_f", "rouge2_f", "rougeL_f",
    "bertscore_p", "bertscore_r", "bertscore_f1",
    "entity_precision", "entity_recall", "entity_f1",
    "compression_ratio", "fk_grade",
) if c in full.columns]

display_cols = [
    "case_id", "compression_in_band",
    *metric_cols, "fk_in_target_band",
    "entity_tp", "entity_fn", "entity_gold",
    "elapsed_s",
]
full[[c for c in display_cols if c in full.columns]].round(3)

,case_id,compression_in_band,rouge1_f,rouge2_f,rougeL_f,bertscore_p,bertscore_r,bertscore_f1,entity_precision,entity_recall,entity_f1,compression_ratio,fk_grade,fk_in_target_band,entity_tp,entity_fn,entity_gold,elapsed_s
0,mimic-72907-165405-520384,False,0.173,0.029,0.092,0.779,0.810,0.794,1.0,0.195,0.327,0.442,14.20,False,8,33,41,8.73
1,mimic-29307-175627-318233,False,0.142,0.044,0.098,0.809,0.818,0.814,1.0,0.130,0.231,0.614,9.26,False,3,20,23,4.27
2,mimic-47927-161682-717576,False,0.107,0.007,0.060,0.785,0.810,0.797,1.0,0.138,0.242,0.454,9.64,False,4,25,29,7.47
3,mimic-77013-141363-444743,False,0.103,0.014,0.050,0.772,0.781,0.777,1.0,0.128,0.227,0.443,8.89,True,5,34,39,8.97
4,mimic-54229-165594-598031,False,0.132,0.032,0.060,0.771,0.778,0.775,1.0,0.250,0.400,0.450,9.38,False,9,27,36,8.47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,mimic-83607-156758-631634,False,0.144,0.013,0.057,0.787,0.793,0.790,1.0,0.093,0.170,0.422,10.56,False,4,39,43,8.52
96,mimic-25326-191645-386337,False,0.074,0.011,0.041,0.785,0.782,0.784,1.0,0.067,0.125,0.741,11.26,False,3,42,45,5.16
97,mimic-56635-192782-481041,False,0.128,0.018,0.062,0.776,0.799,0.787,1.0,0.122,0.218,0.463,10.89,False,6,43,49,13.42
98,mimic-68135-163192-486433,False,0.083,0.014,0.055,0.797,0.784,0.791,1.0,0.061,0.114,0.363,8.98,True,2,31,33,3.86


## Aggregate (mean / median / min / max)

In [50]:
if metric_cols:
    agg = full[metric_cols].agg(["mean", "median", "min", "max"]).round(3)
    display(agg)
else:
    print("No metrics computed.")

,rouge1_f,rouge2_f,rougeL_f,bertscore_p,bertscore_r,bertscore_f1,entity_precision,entity_recall,entity_f1,compression_ratio,fk_grade
mean,0.130,0.020,0.071,0.787,0.799,0.793,1.0,0.151,0.254,0.426,10.222
median,0.128,0.018,0.070,0.787,0.800,0.793,1.0,0.140,0.246,0.421,10.230
min,0.022,0.002,0.008,0.760,0.763,0.768,1.0,0.031,0.061,0.221,5.950
max,0.233,0.047,0.120,0.816,0.834,0.815,1.0,0.368,0.538,0.760,15.190


## Save baseline outputs

In [51]:
csv_path = BASELINE_DIR / "baseline_metrics.csv"
full.to_csv(csv_path, index=False)
print(f"saved per-case metrics to {csv_path}")

if metric_cols:
    agg_path = BASELINE_DIR / "baseline_aggregate.csv"
    agg.to_csv(agg_path)
    print(f"saved aggregate metrics to {agg_path}")

saved per-case metrics to /Users/natejly/Documents/GitHub/LLMS/outputs/_baseline/baseline_metrics.csv
saved aggregate metrics to /Users/natejly/Documents/GitHub/LLMS/outputs/_baseline/baseline_aggregate.csv
